In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv(
    'data/nyc_taxi_2019-07.csv',
    usecols=['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'total_amount'],
    parse_dates=['tpep_pickup_datetime', 'tpep_dropoff_datetime']
)

df

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,total_amount
0,2019-07-01 00:51:04,2019-07-01 00:51:33,1.0,0.00,4.94
1,2019-07-01 00:46:04,2019-07-01 01:05:46,1.0,4.16,20.30
2,2019-07-01 00:25:09,2019-07-01 01:00:56,1.0,18.80,70.67
3,2019-07-01 00:33:32,2019-07-01 01:15:27,1.0,18.46,66.36
4,2019-07-01 00:00:55,2019-07-01 00:13:05,0.0,1.70,15.30
...,...,...,...,...,...
6310414,2019-07-29 16:34:53,2019-07-29 16:53:20,NaN,3.86,29.00
6310415,2019-07-29 16:07:57,2019-07-29 17:26:41,NaN,15.48,54.43
6310416,2019-07-29 16:01:31,2019-07-29 17:15:38,NaN,12.92,65.40
6310417,2019-07-29 16:58:00,2019-07-29 17:42:00,NaN,7.12,43.00


In [3]:
len(df) - df.count()

tpep_pickup_datetime         0
tpep_dropoff_datetime        0
passenger_count          33959
trip_distance                0
total_amount                 0
dtype: int64

In [8]:
df['passenger_count'].value_counts(normalize=True).sort_index() * 100

passenger_count
0.0     1.862260
1.0    69.798740
2.0    15.195763
3.0     4.448065
4.0     2.225936
5.0     4.051042
6.0     2.417127
7.0     0.000462
8.0     0.000351
9.0     0.000255
Name: proportion, dtype: float64

In [9]:
df.dtypes

tpep_pickup_datetime     datetime64[ns]
tpep_dropoff_datetime    datetime64[ns]
passenger_count                 float64
trip_distance                   float64
total_amount                    float64
dtype: object

In [10]:
df['trip_time'] = df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']

In [11]:
df.dtypes

tpep_pickup_datetime      datetime64[ns]
tpep_dropoff_datetime     datetime64[ns]
passenger_count                  float64
trip_distance                    float64
total_amount                     float64
trip_time                timedelta64[ns]
dtype: object

In [12]:
df['trip_time']

0         0 days 00:00:29
1         0 days 00:19:42
2         0 days 00:35:47
3         0 days 00:41:55
4         0 days 00:12:10
                ...      
6310414   0 days 00:18:27
6310415   0 days 01:18:44
6310416   0 days 01:14:07
6310417   0 days 00:44:00
6310418   0 days 00:17:00
Name: trip_time, Length: 6310419, dtype: timedelta64[ns]

In [15]:
len(df.loc[df['trip_time'] < pd.to_timedelta('1 minute')])

70212

In [17]:
len(df.loc[df['trip_time'] < pd.to_timedelta('1 minute')]) / len(df) * 100

1.1126361022936828

In [19]:
df.loc[df['trip_time'] < pd.to_timedelta('1 minute'), 'total_amount'].mean()

np.float64(30.397584031219733)

In [20]:
len(df.loc[df['trip_time'] > pd.to_timedelta('10 hours')])

16698

In [22]:
len(df.loc[df['trip_time'] > pd.to_timedelta('10 hours')]) / len(df) * 100

0.2646100045020782

In [23]:
def get_interval_name(interval):
    if interval < pd.to_timedelta('10 minutes'):
        return 'short'
    elif pd.to_timedelta('10 minutes') <= interval <= pd.to_timedelta('1 hour'):
        return 'medium'
    else:
        return 'long'

In [24]:
df['trip_time_group'] = df['trip_time'].apply(get_interval_name)

In [25]:
df.head()

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,total_amount,trip_time,trip_time_group
0,2019-07-01 00:51:04,2019-07-01 00:51:33,1.0,0.00,4.94,0 days 00:00:29,short
1,2019-07-01 00:46:04,2019-07-01 01:05:46,1.0,4.16,20.30,0 days 00:19:42,medium
2,2019-07-01 00:25:09,2019-07-01 01:00:56,1.0,18.80,70.67,0 days 00:35:47,medium
3,2019-07-01 00:33:32,2019-07-01 01:15:27,1.0,18.46,66.36,0 days 00:41:55,medium
4,2019-07-01 00:00:55,2019-07-01 00:13:05,0.0,1.70,15.30,0 days 00:12:10,medium


In [30]:
(df['trip_time_group'].value_counts(normalize=True) * 100).round(2)

trip_time_group
medium    55.34
short     43.45
long       1.21
Name: proportion, dtype: float64

In [31]:
df.groupby(by=['trip_time_group'])['passenger_count'].mean()

trip_time_group
long      1.700859
medium    1.585768
short     1.551222
Name: passenger_count, dtype: float64

In [34]:
df['trip_time_group_v2'] = pd.cut(df['trip_time'], bins=[pd.to_timedelta(v) for v in ['0 seconds', '10 minutes', '1 hour', '100 hours']], labels=['short', 'medium', 'long'])

In [35]:
df['trip_time'].apply(get_interval_name)

0           short
1          medium
2          medium
3          medium
4          medium
            ...  
6310414    medium
6310415      long
6310416      long
6310417    medium
6310418    medium
Name: trip_time, Length: 6310419, dtype: object

In [36]:
df.groupby(by=['trip_time_group', 'trip_time_group_v2'])['trip_time'].count()

/var/folders/ly/4m8_p07j62l8tfy4gs_lj7100000gn/T/ipykernel_2076/3052037821.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(by=['trip_time_group', 'trip_time_group_v2'])['trip_time'].count()


trip_time_group  trip_time_group_v2
long             short                       0
                 medium                      0
                 long                    76551
medium           short                    5437
                 medium                3486620
                 long                        0
short            short                 2733588
                 medium                      0
                 long                        0
Name: trip_time, dtype: int64

In [37]:
# 39.1

In [45]:
len(df.loc[~df['tpep_pickup_datetime'].between('2019-07-01', '2019-08-01', inclusive='left')])

285

In [46]:
# 39.2

In [47]:
df.groupby(by='passenger_count')['trip_time'].mean()

passenger_count
0.0   0 days 00:14:18.929810752
1.0   0 days 00:17:46.148103924
2.0   0 days 00:18:34.024342704
3.0   0 days 00:19:02.079604271
4.0   0 days 00:20:10.057290100
5.0   0 days 00:22:29.870464324
6.0   0 days 00:20:54.109564300
7.0   0 days 00:16:38.206896551
8.0      0 days 00:11:00.500000
9.0      0 days 00:49:16.125000
Name: trip_time, dtype: timedelta64[ns]

In [49]:
df.loc[df['passenger_count'].isnull(), 'trip_time'].mean()

Timedelta('0 days 00:34:29.161724432')

In [50]:
# 39.3

In [51]:
df1 = pd.read_csv(
    'data/nyc_taxi_2019-07.csv',
    usecols=['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'total_amount'],
    parse_dates=['tpep_pickup_datetime', 'tpep_dropoff_datetime']
)

df1

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,total_amount
0,2019-07-01 00:51:04,2019-07-01 00:51:33,1.0,0.00,4.94
1,2019-07-01 00:46:04,2019-07-01 01:05:46,1.0,4.16,20.30
2,2019-07-01 00:25:09,2019-07-01 01:00:56,1.0,18.80,70.67
3,2019-07-01 00:33:32,2019-07-01 01:15:27,1.0,18.46,66.36
4,2019-07-01 00:00:55,2019-07-01 00:13:05,0.0,1.70,15.30
...,...,...,...,...,...
6310414,2019-07-29 16:34:53,2019-07-29 16:53:20,NaN,3.86,29.00
6310415,2019-07-29 16:07:57,2019-07-29 17:26:41,NaN,15.48,54.43
6310416,2019-07-29 16:01:31,2019-07-29 17:15:38,NaN,12.92,65.40
6310417,2019-07-29 16:58:00,2019-07-29 17:42:00,NaN,7.12,43.00


In [52]:
df2 = pd.read_csv(
    'data/nyc_taxi_2020-07.csv',
    usecols=['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'total_amount'],
    parse_dates=['tpep_pickup_datetime', 'tpep_dropoff_datetime']
)

df2

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,total_amount
0,2020-07-01 00:25:32,2020-07-01 00:33:39,1.0,1.50,9.30
1,2020-07-01 00:03:19,2020-07-01 00:25:43,1.0,9.50,27.80
2,2020-07-01 00:15:11,2020-07-01 00:29:24,1.0,5.85,22.30
3,2020-07-01 00:30:49,2020-07-01 00:38:26,1.0,1.90,14.16
4,2020-07-01 00:31:26,2020-07-01 00:38:02,1.0,1.25,7.80
...,...,...,...,...,...
800407,2020-07-19 13:27:52,2020-07-19 14:22:15,NaN,24.23,83.50
800408,2020-07-19 13:02:00,2020-07-19 13:21:00,NaN,4.40,19.78
800409,2020-07-19 13:32:00,2020-07-19 13:51:00,NaN,8.78,38.45
800410,2020-07-19 13:28:00,2020-07-19 13:51:00,NaN,6.50,29.77


In [54]:
df3 = pd.concat(objs=[df1, df2], axis=0, ignore_index=True)
df3

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,total_amount
0,2019-07-01 00:51:04,2019-07-01 00:51:33,1.0,0.00,4.94
1,2019-07-01 00:46:04,2019-07-01 01:05:46,1.0,4.16,20.30
2,2019-07-01 00:25:09,2019-07-01 01:00:56,1.0,18.80,70.67
3,2019-07-01 00:33:32,2019-07-01 01:15:27,1.0,18.46,66.36
4,2019-07-01 00:00:55,2019-07-01 00:13:05,0.0,1.70,15.30
...,...,...,...,...,...
7110826,2020-07-19 13:27:52,2020-07-19 14:22:15,NaN,24.23,83.50
7110827,2020-07-19 13:02:00,2020-07-19 13:21:00,NaN,4.40,19.78
7110828,2020-07-19 13:32:00,2020-07-19 13:51:00,NaN,8.78,38.45
7110829,2020-07-19 13:28:00,2020-07-19 13:51:00,NaN,6.50,29.77


In [55]:
df3['pickup_year'] = df3['tpep_pickup_datetime'].dt.year

In [56]:
df3.groupby(by=['pickup_year', 'passenger_count'])['total_amount'].mean()

pickup_year  passenger_count
2002         1.0                18.002500
             2.0                18.800000
2008         1.0                18.340000
             2.0                42.860000
             5.0                11.966667
2009         1.0                23.923571
             2.0                45.316000
2010         2.0                18.360000
2019         0.0                18.981793
             1.0                19.284646
             2.0                20.097442
             3.0                20.208111
             4.0                21.063172
             5.0                19.419311
             6.0                19.386516
             7.0                70.080690
             8.0                74.760455
             9.0                93.509375
2020         0.0                16.538912
             1.0                16.856554
             2.0                17.188322
             3.0                17.103106
             4.0                17.964939
     

In [57]:
len(df.loc[~df['tpep_pickup_datetime'].between('2019-07-01', '2019-08-01', inclusive='left')])

285

In [59]:
df.loc[(df['tpep_pickup_datetime'] < '2019-07-01') | (df['tpep_pickup_datetime'] > '2019-07-31 23:59')]

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,total_amount,trip_time,trip_time_group,trip_time_group_v2
184,2019-06-30 14:54:49,2019-06-30 15:04:50,1.0,1.71,13.30,0 days 00:10:01,medium,medium
185,2019-06-30 15:19:34,2019-06-30 15:37:32,1.0,7.05,26.30,0 days 00:17:58,medium,medium
206,2019-06-30 23:41:12,2019-06-30 23:48:54,1.0,1.00,10.30,0 days 00:07:42,short,short
274,2019-06-30 23:52:06,2019-07-01 00:26:02,1.0,10.83,39.90,0 days 00:33:56,medium,medium
421,2019-06-30 23:56:48,2019-07-01 00:03:34,1.0,1.57,10.30,0 days 00:06:46,short,short
...,...,...,...,...,...,...,...,...
6276049,2019-07-31 23:59:22,2019-08-01 00:04:29,4.0,0.82,9.30,0 days 00:05:07,short,short
6276069,2019-08-01 00:04:50,2019-08-01 00:13:48,1.0,2.46,12.80,0 days 00:08:58,short,short
6276128,2019-07-31 23:59:22,2019-08-01 00:05:27,1.0,0.80,9.80,0 days 00:06:05,short,short
6276258,2019-07-31 23:59:05,2019-08-01 00:07:34,1.0,1.90,14.75,0 days 00:08:29,short,short
